In [9]:
#import necessary libraries and load datasets
import pandas as pd

sentiment = pd.read_csv("fear_greed_index.csv")
trades = pd.read_csv("historical_data.csv")

In [ ]:
# print dataset shapes
print("Sentiment dataset shape:", sentiment.shape)
print("Trades dataset shape:", trades.shape)

Sentiment dataset shape: (2644, 4)
Trades dataset shape: (211224, 16)


In [9]:
# Check for missing values
print("sentiment data missing values:\n", sentiment.isnull().sum())
print("trades data missing values:\n", trades.isnull().sum())

sentiment data missing values:
 timestamp         0
value             0
classification    0
date              0
dtype: int64
trades data missing values:
 Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64


In [ ]:
#check for duplicates
sentiment.duplicated().sum()
trades.duplicated().sum()

np.int64(0)

In [17]:
trades.dtypes

Account              object
Coin                 object
Execution Price     float64
Size Tokens         float64
Size USD            float64
Side                 object
Timestamp IST        object
Start Position      float64
Direction            object
Closed PnL          float64
Transaction Hash     object
Order ID              int64
Crossed                bool
Fee                 float64
Trade ID            float64
Timestamp           float64
dtype: object

In [18]:
sentiment.dtypes

timestamp          int64
value              int64
classification    object
date              object
dtype: object

In [26]:
#converting timestamps
sentiment['datetime'] = pd.to_datetime(sentiment['timestamp'], unit='s')
sentiment['date'] = sentiment['datetime'].dt.date
sentiment[['timestamp','datetime','date']].head()

,timestamp,datetime,date
0,1517463000,2018-02-01 05:30:00,2018-02-01
1,1517549400,2018-02-02 05:30:00,2018-02-02
2,1517635800,2018-02-03 05:30:00,2018-02-03
3,1517722200,2018-02-04 05:30:00,2018-02-04
4,1517808600,2018-02-05 05:30:00,2018-02-05


In [ ]:
#converting timestamps
trades['datetime'] = pd.to_datetime(trades['Timestamp'], unit='ms')
trades['date'] = trades['datetime'].dt.date

trades[['Timestamp','datetime','date']].head()

,Timestamp,datetime,date
0,1.730000e+12,2024-10-27 03:33:20,2024-10-27
1,1.730000e+12,2024-10-27 03:33:20,2024-10-27
2,1.730000e+12,2024-10-27 03:33:20,2024-10-27
3,1.730000e+12,2024-10-27 03:33:20,2024-10-27
4,1.730000e+12,2024-10-27 03:33:20,2024-10-27


In [ ]:
#merging datasets on date
merged = pd.merge(trades, sentiment[['date','classification']], on='date', how='left')
merged[['date','classification']].head()

,date,classification
0,2024-10-27,Greed
1,2024-10-27,Greed
2,2024-10-27,Greed
3,2024-10-27,Greed
4,2024-10-27,Greed


In [ ]:
# Calculate daily PnL for each account
daily_pnl = trades.groupby(['Account','date'])['Closed PnL'].sum().reset_index()
daily_pnl.head()

,Account,date,Closed PnL
0,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-10-27,-3.275059e+05
1,0x083384f897ee0f19899168e3b1bec365f52a9012,2025-02-19,1.927736e+06
2,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,2024-10-27,2.060745e+04
3,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,2025-02-19,1.709873e+04
4,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,2025-06-15,1.017915e+04


In [ ]:
# Calculate win rate for each account
trades['win'] = trades['Closed PnL'] > 0

win_rate = trades.groupby('Account')['win'].mean().reset_index()
win_rate['win_rate'] = win_rate['win'] * 100
win_rate[['Account','win_rate']].head()

,Account,win_rate
0,0x083384f897ee0f19899168e3b1bec365f52a9012,35.961236
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,44.271978
2,0x271b280974205ca63b716753467d5a371de622ab,30.191651
3,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,43.858463
4,0x2c229d22b100a7beb69122eed721cee9b24011dd,51.991355


In [46]:
# Calculate win rate for each day
daily_win_rate = trades.groupby('date')['win'].mean().reset_index()
daily_win_rate['win_rate'] = daily_win_rate['win'] * 100
daily_win_rate[['date','win_rate']].head()


,date,win_rate
0,2023-03-28,0.000000
1,2023-11-14,27.464115
2,2024-03-09,49.008905
3,2024-07-03,31.718247
4,2024-10-27,45.160467


In [ ]:
# Calculate average trade size for each account
avg_trade_size = trades.groupby('Account')['Size USD'].mean().reset_index()
avg_trade_size.head()

,Account,Size USD
0,0x083384f897ee0f19899168e3b1bec365f52a9012,16159.576734
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,1653.226327
2,0x271b280974205ca63b716753467d5a371de622ab,8893.000898
3,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,507.626933
4,0x2c229d22b100a7beb69122eed721cee9b24011dd,3138.894782


In [47]:
# Calculate average trade size for each day
daily_avg_trade_size = trades.groupby('date')['Size USD'].mean().reset_index()
daily_avg_trade_size.head()

,date,Size USD
0,2023-03-28,159.000000
1,2023-11-14,11057.827522
2,2024-03-09,5660.265764
3,2024-07-03,3058.848110
4,2024-10-27,2949.625864


In [48]:
#no. of trades per day
trades_per_day = trades.groupby('date').size().reset_index(name='num_trades')
trades_per_day.head()

,date,num_trades
0,2023-03-28,3
1,2023-11-14,1045
2,2024-03-09,6962
3,2024-07-03,7141
4,2024-10-27,35241


In [45]:
#long vs short analysis
open_positions = trades[trades['Direction'].isin(['Open Long','Open Short'])]
long_short_counts = open_positions['Direction'].value_counts()
long_short_ratio = long_short_counts['Open Long'] / long_short_counts['Open Short']
print("Long vs Short Ratio:", long_short_ratio)
long_short_percentage = open_positions['Direction'].value_counts(normalize=True)*100
print("\nLong vs Short Percentage:\n", long_short_percentage)

Long vs Short Ratio: 1.25550439093128

Long vs Short Percentage:
 Direction
Open Long     55.664019
Open Short    44.335981
Name: proportion, dtype: float64


In [38]:
trades_per_day = trades.groupby('date').size().reset_index(name='num_trades')
trades_per_day.head()

,date,num_trades
0,2023-03-28,3
1,2023-11-14,1045
2,2024-03-09,6962
3,2024-07-03,7141
4,2024-10-27,35241
